# Water Quality Prediction: Benchmark Notebook 

## Challenge Overview

Welcome to the EY AI & Data Challenge 2026!  
The objective of this challenge is to build a robust **machine learning model** capable of predicting water quality across various river locations in South Africa. In addition to accurate predictions, the model should also identify and emphasize the key factors that significantly influence water quality.

Participants will be provided with a dataset containing three water quality parameters — **Total Alkalinity**, **Electrical Conductance**, and **Dissolved Reactive Phosphorus** — collected between 2011 and 2015 from approximately 200 river locations across South Africa. Each data point includes the geographic coordinates (latitude and longitude) of the sampling site, the date of collection, and the corresponding water quality measurements.

Using this dataset, participants are expected to build a machine learning model to predict water quality parameters for a separate validation dataset, which includes locations from different regions not present in the training data. The challenge also encourages participants to explore feature importance and provide insights into the factors most strongly associated with variations in water quality.

This challenge is designed for participants with varying levels of experience in data science, remote sensing, and environmental analytics. It offers a valuable opportunity to apply machine learning techniques to real-world environmental data and contribute to advancing water quality monitoring using artificial intelligence.

**About the Notebook:**  

In this notebook, we demonstrate a basic workflow that serves as a foundation for the challenge. The model has been developed to predict **water quality parameters** using features derived from the **Landsat** and **TerraClimate** datasets. Specifically, four spectral bands — **SWIR22** (Shortwave Infrared 2), **NIR** (Near Infrared), **Green**, and **SWIR16** (Shortwave Infrared 1) — were utilized from Landsat, along with derived spectral indices such as **NDMI** (Normalized Difference Moisture Index) and **MNDWI** (Modified Normalized Difference Water Index). In addition, the **PET** (Potential Evapotranspiration) variable was incorporated from the **TerraClimate** dataset to account for climatic influences on water quality.

The dataset spans a five-year period from **2011 to 2015**. Using **API-based data extraction** methods, both Landsat and TerraClimate features were retrieved directly from the [Microsoft Planetary Computer portal](https://planetarycomputer.microsoft.com).

These combined spectral, index-based, and climatic features were used as predictors in a regression model to estimate three key water quality parameters: **Total Alkalinity (TA)**, **Electrical Conductance (EC)**, and **Dissolved Reactive Phosphorus (DRP)**.

Please note that this notebook serves only as a starting point. Several assumptions were made during the data extraction and model development process, which you may find opportunities to improve upon. Participants are encouraged to explore additional features, enhance preprocessing techniques, or experiment with different regression algorithms to optimize predictive performance.

## Load In Dependencies
The following code installs the required Python libraries (found in the requirements.txt file) in the Snowflake environment to allow successful execution of the remaining notebook code. After running this code for the first time, it is required to “restart” the kernal so the Python libraries are available in the environment. This is done by selecting the “Connected” menu above the notebook (next to “Run all”) and selecting the “restart kernal” link. Subsequent runs of the notebook do not require this “restart” process.

In [1]:
!pip install uv
!uv pip install -r requirements.txt xgboost


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
Resolved 118 packages in 852ms
 Downloaded xgboost
Prepared 1 package in 1.18s
Installed 1 package in 23ms
 + xgboost==3.2.0


In [2]:
# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Data manipulation and analysis
import numpy as np
import pandas as pd
from IPython.display import display

# Multi-dimensional arrays and datasets (e.g., NetCDF, Zarr)
import xarray as xr

# Geospatial raster data handling with CRS support
import rioxarray as rxr

# Raster operations and spatial windowing
import rasterio
from rasterio.windows import Window

# Feature preprocessing and data splitting
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GroupKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from scipy.spatial import cKDTree
from scipy.stats import uniform, randint

# Machine Learning
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error, make_scorer

# Planetary Computer tools for STAC API access and authentication
import pystac_client
import planetary_computer as pc
from odc.stac import stac_load
from pystac.extensions.eo import EOExtension as eo

from datetime import date
from tqdm import tqdm
import os

## Response Variable

Before building the model, we first load the **water quality training dataset**. The curated dataset contains samples collected from various monitoring stations across the study region. Each record includes the geographical coordinates (Latitude and Longitude), the sample collection date, and the corresponding **measured values** for the three key water quality parameters — **Total Alkalinity (TA)**, **Electrical Conductance (EC)**, and **Dissolved Reactive Phosphorus (DRP)**.

In [3]:
Water_Quality_df = pd.read_csv("water_quality_training_dataset.csv")
display(Water_Quality_df.head(5))

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-28.760833,17.730278,02-01-2011,128.912,555.0,10.0
1,-26.861111,28.884722,03-01-2011,74.720,162.9,163.0
2,-26.450000,28.085833,03-01-2011,89.254,573.0,80.0
3,-27.671111,27.236944,03-01-2011,82.000,203.6,101.0
4,-27.356667,27.286389,03-01-2011,56.100,145.1,151.0


## Predictor Variables

Now that we have our water quality dataset, the next step is to gather the predictor variables from the **Landsat** and **TerraClimate** datasets. In this notebook, we demonstrate how to **load previously extracted satellite and climate data** from separate files, rather than performing the extraction directly, which allows for a smoother and faster experience. Participants can refer to the dedicated extraction notebooks—one for Landsat and another for TerraClimate—to understand how the data was retrieved and processed, and they can also generate their own output CSV files if needed. Using these pre-extracted CSV files, this notebook focuses on loading the predictor features and running the subsequent analysis and model training efficiently.

For more detailed guidance on the original data extraction process, you can review the Landsat and TerraClimate example notebooks available on the Planetary Computer portal:

- [Landsat-c2-l2 - Example-Notebook](https://planetarycomputer.microsoft.com/dataset/landsat-c2-l2#Example-Notebook)  
- [Terraclimate - Example-Notebook](https://planetarycomputer.microsoft.com/dataset/terraclimate#Example-Notebook)

We have used selected spectral bands — **SWIR22** (Shortwave Infrared 2), **NIR** (Near Infrared), **Green**, and **SWIR16** (Shortwave Infrared 1) — and computed key spectral indices such as **NDMI** (Normalized Difference Moisture Index) and **MNDWI** (Modified Normalized Difference Water Index). These features capture surface moisture, vegetation, and water content characteristics that influence water quality variability.

In addition to Landsat features, we also incorporated the **Potential Evapotranspiration (PET)** variable from the **TerraClimate** dataset, which provides high-resolution global climate data. The PET feature captures the atmospheric demand for moisture, representing climatic conditions such as temperature, humidity, and radiation that influence surface water evaporation and thus affect water quality parameters.

The predictor features include:

- **SWIR22** – Sensitive to surface moisture and turbidity variations in water bodies.  
- **NIR** – Helps in identifying vegetation and suspended matter in water.  
- **Green** – Useful for detecting water color and surface reflectance changes.  
- **SWIR16** – Provides information on surface dryness and sediment concentration.  
- **NDMI** – Derived from NIR and SWIR16, indicates moisture and vegetation–water interaction.  
- **MNDWI** – Derived from Green and SWIR22, effective for distinguishing open water areas and reducing built-up noise.  
- **PET** – Extracted from the TerraClimate dataset, represents potential evapotranspiration influencing hydrological and water quality dynamics.

### **Tip 1**

Participants are encouraged to experiment with different combinations of **Landsat** bands or even include data from other public satellite data sources. By creating mathematical combinations of bands, you can derive various spectral indices that capture surface and environmental characteristics.

### Loading Pre-Extracted Landsat Data

In this notebook, we **load previously extracted Landsat data** from CSV files generated in a separate extraction notebook. This approach ensures a smoother and faster workflow, allowing participants to focus on data analysis and model development without waiting for time-consuming data retrieval.

Participants are expected to generate their own data extraction CSV files by running the dedicated Landsat extraction notebook. These CSV files can then be used here to smoothly run this benchmark notebook. Participants can refer to the extraction notebook to understand the API-based process, including how individual bands and indices like **NDMI** were computed. Using these pre-extracted CSV files simplifies preprocessing and is ideal for large-scale environmental and water quality analysis.

### **Tip 2**

In the data extraction process (performed in the dedicated extraction notebooks), a 100 m focal buffer was applied around each sampling location rather than using a single point. Participants may explore creating different focal buffers around the locations (e.g., 50 m, 150 m, etc.) during extraction. For example, if a 50 m buffer was used for “Band 2”, the extracted CSV values would reflect the average of Band 2 within 50 meters of each location. This approach can help reduce errors associated with spatial autocorrelation.

In [4]:
landsat_train_features = pd.read_csv("landsat_features_training.csv")
display(landsat_train_features.head(5))

,Latitude,Longitude,Sample Date,nir,green,swir16,swir22,NDMI,MNDWI
0,-28.760833,17.730278,02-01-2011,11190.0,11426.0,7687.5,7645.0,0.185538,0.195595
1,-26.861111,28.884722,03-01-2011,17658.5,9550.0,13746.5,10574.0,0.124566,-0.180134
2,-26.450000,28.085833,03-01-2011,15210.0,10720.0,17974.0,14201.0,-0.083293,-0.252805
3,-27.671111,27.236944,03-01-2011,14887.0,10943.0,13522.0,11403.0,0.048048,-0.105416
4,-27.356667,27.286389,03-01-2011,16828.5,9502.5,12665.5,9643.0,0.141147,-0.142683


In [5]:
# If NDMI and MNDWI columns are of type object, convert them to float
landsat_train_features['NDMI'] = landsat_train_features['NDMI'].astype(float)
landsat_train_features['MNDWI'] = landsat_train_features['MNDWI'].astype(float)

### Loading Pre-Extracted TerraClimate Data

In this notebook, we **load previously extracted TerraClimate data** from CSV files generated in a dedicated extraction notebook. This approach ensures a smoother and faster workflow, allowing participants to focus on data analysis and model development without waiting for time-consuming data retrieval.

Participants are expected to generate their own data extraction CSV files by running the dedicated TerraClimate extraction notebook. These CSV files can then be used here to smoothly run this benchmark notebook. Participants can refer to the extraction notebook to understand the API-based process, including how climate variables such as **Potential Evapotranspiration (PET)** were extracted. Using these pre-extracted CSV files ensures consistent, automated retrieval of high-resolution climate data that can be easily integrated with satellite-derived features for comprehensive environmental and hydrological analysis.

In [6]:
Terraclimate_df = pd.read_csv("terraclimate_features_training.csv")
display(Terraclimate_df.head(5))

,Latitude,Longitude,Sample Date,pet
0,-28.760833,17.730278,02-01-2011,174.2
1,-26.861111,28.884722,03-01-2011,124.1
2,-26.450000,28.085833,03-01-2011,127.5
3,-27.671111,27.236944,03-01-2011,129.7
4,-27.356667,27.286389,03-01-2011,129.2


## Joining the Predictor Variables and Response Variables

Now that we have extracted our predictor variables, we need to join them with the response variables. We use the **combine_two_datasets** function to merge the predictor variables and response variables. The **concat** function from pandas is particularly useful for this step.

In [7]:
# Combine two datasets vertically (along columns) using pandas concat function.
def combine_two_datasets(dataset1,dataset2,dataset3):
    '''
    Returns a  vertically concatenated dataset.
    Attributes:
    dataset1 - Dataset 1 to be combined 
    dataset2 - Dataset 2 to be combined
    '''
    
    data = pd.concat([dataset1,dataset2,dataset3], axis=1)
    data = data.loc[:, ~data.columns.duplicated()]
    return data

In [8]:
# Combining ground data and final data into a single dataset.
wq_data = combine_two_datasets(Water_Quality_df, landsat_train_features, Terraclimate_df)
display(wq_data.head(5))

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus,nir,green,swir16,swir22,NDMI,MNDWI,pet
0,-28.760833,17.730278,02-01-2011,128.912,555.0,10.0,11190.0,11426.0,7687.5,7645.0,0.185538,0.195595,174.2
1,-26.861111,28.884722,03-01-2011,74.720,162.9,163.0,17658.5,9550.0,13746.5,10574.0,0.124566,-0.180134,124.1
2,-26.450000,28.085833,03-01-2011,89.254,573.0,80.0,15210.0,10720.0,17974.0,14201.0,-0.083293,-0.252805,127.5
3,-27.671111,27.236944,03-01-2011,82.000,203.6,101.0,14887.0,10943.0,13522.0,11403.0,0.048048,-0.105416,129.7
4,-27.356667,27.286389,03-01-2011,56.100,145.1,151.0,16828.5,9502.5,12665.5,9643.0,0.141147,-0.142683,129.2


### Handling Missing Values

Before model training, missing values in the dataset were carefully handled to ensure data consistency and prevent model bias. Numerical columns were imputed using their median values, maintaining the overall data distribution while minimizing the impact of outliers.

In [9]:
wq_data = wq_data.fillna(wq_data.median(numeric_only=True))
wq_data.isna().sum()

Latitude                         0
Longitude                        0
Sample Date                      0
Total Alkalinity                 0
Electrical Conductance           0
Dissolved Reactive Phosphorus    0
nir                              0
green                            0
swir16                           0
swir22                           0
NDMI                             0
MNDWI                            0
pet                              0
dtype: int64

## Model Building

We now use **all 7 available features** — the 4 Landsat bands (**NIR**, **Green**, **SWIR16**, **SWIR22**), the 2 derived indices (**NDMI**, **MNDWI**), and **PET** from TerraClimate. The benchmark only used 4, discarding NIR, Green, and SWIR16.

We also apply a **log1p transform** to the target variables before training. Water quality parameters are right-skewed and span wide ranges, so modeling in log-space helps XGBoost learn more stable patterns. Predictions are reverted with `expm1` before submission.

In [10]:
# All 7 features + Latitude/Longitude (for GroupKFold groups only, not used as predictors)
FEATURE_COLS = ['nir', 'green', 'swir16', 'swir22', 'NDMI', 'MNDWI', 'pet']

wq_data = wq_data[['Latitude', 'Longitude'] + FEATURE_COLS +
                   ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']]

### **Tip 3**

However, participants are encouraged to experiment with different feature combinations to build more robust machine learning models.

## Helper Functions

### Pipeline with StandardScaler + XGBoost
We use a scikit-learn `Pipeline` that wraps the `StandardScaler` and the `XGBRegressor` into a single object. This **prevents data leakage** because the scaler is fitted only on the training data of each fold during cross-validation, never seeing the validation data.

### GroupKFold by Location
Instead of a simple random split, we use **GroupKFold** grouping by geographic location (combination of Latitude and Longitude). This ensures that **all samples from the same monitoring station stay in the same fold**, simulating real spatial evaluation — the model is evaluated on locations it has never seen during training.

The locations (Latitude, Longitude) are used only to define the fold splits during GroupKFold — they are not used as input features to the model.

So, the groups parameter tells GroupKFold which rows belong together (same station), but the model itself only ever sees swir22, NDMI, MNDWI, and pet. The groups ensure that during hyperparameter tuning, no fold leaks data from a station it's supposed to evaluate on — but the trained model has no knowledge of coordinates.

At final prediction time, pipeline_TA.predict(submission_val_data) receives only ['swir22', 'NDMI', 'MNDWI', 'pet'] — no location information at all.

### RandomizedSearchCV for Hyperparameter Tuning
We use `RandomizedSearchCV` to efficiently search for the best XGBoost hyperparameters by sampling random combinations from the search space. This is faster than an exhaustive `GridSearchCV` and generally finds good configurations with fewer iterations.

### Model Evaluation
After tuning, we evaluate the best model found using:
- **R² Score**: Measures how well the model explains the variance in the observed values.
- **RMSE (Root Mean Square Error)**: Quantifies the average magnitude of prediction errors.

### **Tip 4**

There are many data preprocessing methods available that may help improve model performance. Participants are encouraged to explore various preprocessing techniques as well as different machine learning algorithms to build a more robust model.

In [11]:
def create_location_groups(wq_data):
    """Create group labels based on unique (Latitude, Longitude) pairs for GroupKFold."""
    groups = wq_data[['Latitude', 'Longitude']].apply(lambda row: f"{row['Latitude']}_{row['Longitude']}", axis=1)
    return groups

def build_pipeline():
    """Build a Pipeline with StandardScaler + XGBRegressor (avoids data leakage)."""
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', XGBRegressor(
            objective='reg:squarederror',
            random_state=42,
            n_jobs=-1,
            verbosity=0
        ))
    ])
    return pipeline

def get_xgb_param_distributions():
    """Define hyperparameter search space for XGBoost."""
    param_distributions = {
        'model__n_estimators':      randint(200, 800),
        'model__max_depth':         [4, 6, 8],       # mínimo 4, não 3
        'model__learning_rate':     uniform(0.05, 0.15),  # 0.05 a 0.20
        'model__subsample':         uniform(0.6, 0.4),
        'model__colsample_bytree':  uniform(0.6, 0.4),
        'model__min_child_weight':  [1, 2, 3],        # máximo 3
        'model__reg_alpha':         [0, 0.01, 0.1],   # quase zero
        'model__reg_lambda':        [0.5, 1.0, 1.5],  # padrão é 1
        'model__gamma':             [0, 0.01, 0.05],  # quase zero
    }
    return param_distributions

def evaluate_predictions(y_true, y_pred, dataset_name="Test"):
    """Evaluate predictions with R² and RMSE."""
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"\n{dataset_name} Evaluation:")
    print(f"  R²:   {r2:.4f}")
    print(f"  RMSE: {rmse:.4f}")
    return r2, rmse

## Model Workflow (Pipeline)

The complete pipeline follows a robust structure to ensure **reproducibility**, **no data leakage**, and **realistic spatial evaluation**:

1. **Location-based group creation** — each unique (Latitude, Longitude) combination receives a group label.
2. **GroupKFold (5 folds)** — ensures that samples from the same station never appear simultaneously in training and validation.
3. **Pipeline (StandardScaler → XGBRegressor)** — the scaler is fitted only within each training fold.
4. **RandomizedSearchCV** — explores 50 random hyperparameter combinations, evaluating each with GroupKFold.
5. **Final evaluation** — the best model is refitted on all data and also evaluated via cross-validation scores.

In [12]:
def run_pipeline(X, y_log, groups, param_name="Parameter", n_splits=5, n_iter=50):
    print(f"\n{'='*60}")
    print(f"Training Model for {param_name}")
    print(f"{'='*60}")
    
    group_kfold = GroupKFold(n_splits=n_splits)
    pipeline = build_pipeline()
    param_distributions = get_xgb_param_distributions()
    
    # Scorers that convert back to original scale before computing metrics
    rmse_scorer = make_scorer(
        lambda y_true, y_pred: np.sqrt(mean_squared_error(np.expm1(y_true), np.expm1(y_pred))),
        greater_is_better=False
    )
    r2_scorer = make_scorer(
        lambda y_true, y_pred: r2_score(np.expm1(y_true), np.expm1(y_pred)),
        greater_is_better=True
    )
    
    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_distributions,
        n_iter=n_iter,
        cv=group_kfold,
        scoring={'r2': r2_scorer, 'rmse': rmse_scorer},
        refit='r2',
        random_state=42,
        n_jobs=-1,
        verbose=1,
        return_train_score=True
    )
    
    search.fit(X, y_log, groups=groups)
    
    best_pipeline = search.best_estimator_
    best_idx = search.best_index_
    cv = search.cv_results_
    
    r2_train = cv['mean_train_r2'][best_idx]
    r2_val   = cv['mean_test_r2'][best_idx]
    rmse_train = -cv['mean_train_rmse'][best_idx]
    rmse_val   = -cv['mean_test_rmse'][best_idx]
    
    print(f"\nBest Hyperparameters:")
    for param, value in search.best_params_.items():
        print(f"  {param}: {value}")
    
    print(f"\nCross-Validation Results — Original Scale (GroupKFold, {n_splits} folds):")
    print(f"  Train R²:  {r2_train:.4f}  |  Train RMSE: {rmse_train:.4f}")
    print(f"  Val   R²:  {r2_val:.4f}  |  Val   RMSE: {rmse_val:.4f}")
    
    results = {
        "Parameter": param_name,
        "CV_R2_Train": round(r2_train, 4),
        "CV_RMSE_Train": round(rmse_train, 4),
        "CV_R2_Val": round(r2_val, 4),
        "CV_RMSE_Val": round(rmse_val, 4),
    }
    
    return best_pipeline, pd.DataFrame([results])

### Model Training and Evaluation for Each Parameter

In this step, we apply the complete modeling pipeline to each of the three selected water quality parameters — Total Alkalinity, Electrical Conductance, and Dissolved Reactive Phosphorus. 

**Importantly**, we create **location-based groups** from the (Latitude, Longitude) pairs and pass them to `GroupKFold`, so the model is always validated on locations it has never seen during training. The `RandomizedSearchCV` explores 50 random hyperparameter combinations for XGBoost, evaluated across 5 spatial folds.

In [13]:
# Create location-based groups for GroupKFold
groups = create_location_groups(wq_data)
print(f"Total samples: {len(wq_data)}")
print(f"Unique locations (groups): {groups.nunique()}")

# Feature matrix — all 7 features
X = wq_data[FEATURE_COLS]

# Target variables with log1p transform (right-skewed → more stable training)
y_TA_log = np.log1p(wq_data['Total Alkalinity'])
y_EC_log = np.log1p(wq_data['Electrical Conductance'])
y_DRP_log = np.log1p(wq_data['Dissolved Reactive Phosphorus'])

# Run pipeline for each parameter (trained on log-space targets)
pipeline_TA, results_TA = run_pipeline(X, y_TA_log, groups, "Total Alkalinity (log)")
pipeline_EC, results_EC = run_pipeline(X, y_EC_log, groups, "Electrical Conductance (log)")
pipeline_DRP, results_DRP = run_pipeline(X, y_DRP_log, groups, "Dissolved Reactive Phosphorus (log)")

Total samples: 9319
Unique locations (groups): 162

Training Model for Total Alkalinity (log)
Fitting 5 folds for each of 50 candidates, totalling 250 fits

Best Hyperparameters:
  model__colsample_bytree: 0.749816047538945
  model__gamma: 0
  model__learning_rate: 0.07751521847992457
  model__max_depth: 4
  model__min_child_weight: 1
  model__n_estimators: 321
  model__reg_alpha: 0.1
  model__reg_lambda: 1.5
  model__subsample: 0.6232334448672797

Cross-Validation Results — Original Scale (GroupKFold, 5 folds):
  Train R²:  0.5007  |  Train RMSE: 52.7081
  Val   R²:  -0.1153  |  Val   RMSE: 77.6102

Training Model for Electrical Conductance (log)
Fitting 5 folds for each of 50 candidates, totalling 250 fits

Best Hyperparameters:
  model__colsample_bytree: 0.749816047538945
  model__gamma: 0
  model__learning_rate: 0.07751521847992457
  model__max_depth: 4
  model__min_child_weight: 1
  model__n_estimators: 321
  model__reg_alpha: 0.1
  model__reg_lambda: 1.5
  model__subsample: 0.623

### Model Performance Summary

The table below consolidates the **GroupKFold cross-validation** results for each water quality parameter. The training and validation metrics represent **averages across folds**, where each fold ensures complete spatial separation between training and validation stations.

In [14]:
results_summary = pd.concat([results_TA, results_EC, results_DRP], ignore_index=True)
results_summary

,Parameter,CV_R2_Train,CV_RMSE_Train,CV_R2_Val,CV_RMSE_Val
0,Total Alkalinity (log),0.5007,52.7081,-0.1153,77.6102
1,Electrical Conductance (log),0.5305,233.9912,0.0029,335.2590
2,Dissolved Reactive Phosphorus (log),0.4789,36.7845,-0.0448,51.8308


## Submission (Test Set Predictions)

**Naming clarification:** The provided CSV files use the word "validation" in their filenames (e.g., `landsat_features_validation.csv`), but these are actually the **test set** — we do **not** know the true labels. Our real validation happens internally via `GroupKFold` during training. Below, we rename variables to `test_*` for clarity.

In [15]:
submission_template = pd.read_csv("submission_template.csv")
display(submission_template.head(5))

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-32.043333,27.822778,01-09-2014,NaN,NaN,NaN
1,-33.329167,26.077500,16-09-2015,NaN,NaN,NaN
2,-32.991639,27.640028,07-05-2015,NaN,NaN,NaN
3,-34.096389,24.439167,07-02-2012,NaN,NaN,NaN
4,-32.000556,28.581667,01-10-2014,NaN,NaN,NaN


In [16]:
landsat_test_features = pd.read_csv("landsat_features_validation.csv")  # "validation" file = actual test set
display(landsat_test_features.head(5))

,Latitude,Longitude,Sample Date,nir,green,swir16,swir22,NDMI,MNDWI
0,-32.043333,27.822778,01-09-2014,15229.0,12868.0,14797.0,12421.0,0.014388,-0.069727
1,-33.329167,26.077500,16-09-2015,NaN,NaN,NaN,NaN,NaN,NaN
2,-32.991639,27.640028,07-05-2015,16221.0,9304.5,12536.5,9958.0,0.128123,-0.147979
3,-34.096389,24.439167,07-02-2012,NaN,NaN,NaN,NaN,NaN,NaN
4,-32.000556,28.581667,01-10-2014,9125.0,11100.5,9455.0,8711.0,-0.017761,0.080052


In [17]:
terraclimate_test_df = pd.read_csv("terraclimate_features_validation.csv")  # "validation" file = actual test set
display(terraclimate_test_df.head(5))

,Latitude,Longitude,Sample Date,pet
0,-32.043333,27.822778,01-09-2014,161.90001
1,-33.329167,26.077500,16-09-2015,177.60000
2,-32.991639,27.640028,07-05-2015,158.40001
3,-34.096389,24.439167,07-02-2012,130.00000
4,-32.000556,28.581667,01-10-2014,152.50000


In [18]:
# Consolidate test set features into a single dataframe
test_data = pd.DataFrame({
    'Longitude': landsat_test_features['Longitude'].values,
    'Latitude': landsat_test_features['Latitude'].values,
    'Sample Date': landsat_test_features['Sample Date'].values,
    'nir': landsat_test_features['nir'].values,
    'green': landsat_test_features['green'].values,
    'swir16': landsat_test_features['swir16'].values,
    'swir22': landsat_test_features['swir22'].values,
    'NDMI': landsat_test_features['NDMI'].values,
    'MNDWI': landsat_test_features['MNDWI'].values,
    'pet': terraclimate_test_df['pet'].values,
})

In [19]:
# Impute missing values in the test set
test_data = test_data.fillna(test_data.median(numeric_only=True))

In [20]:
# Select the same 7 features used during training
X_test = test_data[FEATURE_COLS]
display(X_test.head())

,nir,green,swir16,swir22,NDMI,MNDWI,pet
0,15229.0,12868.0,14797.0,12421.0,0.014388,-0.069727,161.90001
1,14525.5,9493.5,12425.5,9973.0,0.081427,-0.130571,177.60000
2,16221.0,9304.5,12536.5,9958.0,0.128123,-0.147979,158.40001
3,14525.5,9493.5,12425.5,9973.0,0.081427,-0.130571,130.00000
4,9125.0,11100.5,9455.0,8711.0,-0.017761,0.080052,152.50000


In [21]:
X_test.shape

(200, 7)

In [22]:
# Predict in log-space, then revert with expm1 to get original scale
pred_TA = np.expm1(pipeline_TA.predict(X_test))
pred_EC = np.expm1(pipeline_EC.predict(X_test))
pred_DRP = np.expm1(pipeline_DRP.predict(X_test))

In [23]:
submission_df = pd.DataFrame({
    'Longitude': submission_template['Longitude'].values,
    'Latitude': submission_template['Latitude'].values,
    'Sample Date': submission_template['Sample Date'].values,
    'Total Alkalinity': pred_TA,
    'Electrical Conductance': pred_EC,
    'Dissolved Reactive Phosphorus': pred_DRP
})

In [24]:
#Displaying the sample submission dataframe
display(submission_df.head())

,Longitude,Latitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,27.822778,-32.043333,01-09-2014,67.383629,205.622025,18.101641
1,26.077500,-33.329167,16-09-2015,124.310341,549.442139,68.002731
2,27.640028,-32.991639,07-05-2015,70.964340,367.888123,29.782810
3,24.439167,-34.096389,07-02-2012,61.893078,225.195694,20.583414
4,28.581667,-32.000556,01-10-2014,95.446808,263.585663,17.428909


In [25]:
#Dumping the predictions into a csv file.
submission_df.to_csv("submission Isaac.csv",index = False)

In [ ]:
session.sql("""
    PUT file:///tmp/submission.csv
    'snow://workspace/USER$.PUBLIC."ey-hackathon"/versions/live/'
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()
print("File saved! Refresh the browser to see the files in the sidebar")

### Upload submission file on platform

Upload the `submission.csv` file on the challenge platform to generate your score on the leaderboard.

## Conclusion

Now that you have learned a basic approach to model training, it’s time to explore your own techniques and ideas! Feel free to modify any of the functions presented in this notebook to experiment with alternative preprocessing steps, feature engineering strategies, or machine learning algorithms. 

We look forward to seeing your enhanced model and the insights you uncover. Best of luck with the challenge!